In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  MedAssist AI — Notebook 2 : LangGraph Stateful Agent
# ═══════════════════════════════════════════════════════════════════

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────

!pip install -q --force-reinstall "numpy==1.26.4" "pandas==2.2.2"

# LangChain ecosystem
!pip install -q \
    "langchain==0.2.16" \
    "langchain-community==0.2.16" \
    "langchain-huggingface==0.0.3" \
    "langgraph==0.2.28"

# Vector store & retrieval
!pip install -q \
    "faiss-cpu==1.8.0" 


# HuggingFace stack
!pip install -q \
    "transformers==4.44.2" \
    "sentence-transformers==3.0.1" \
    "accelerate==0.33.0" \
    "bitsandbytes>=0.43.0"

# Dataset & utilities
!pip install -q \
    "datasets==2.21.0" \
    "tqdm==4.66.5" \
    "scikit-learn" \
    "pydantic>=2.0"

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# ── Cell 2: Imports & global config ──────────────────────────────────
import warnings, json, pickle, re, time
warnings.filterwarnings("ignore")

import torch
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import TypedDict, Optional, List, Annotated
from pydantic import BaseModel, Field
from kaggle_secrets import UserSecretsClient
# HuggingFace
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline,
    StoppingCriteria, StoppingCriteriaList,
)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# LangChain
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

# LangGraph
from langgraph.graph import StateGraph, END
# ── Central config ────────────────────────────────────────────────────
CONFIG = {
    # ── Models ───────────────────────────────────────────────────────
    "embedding_model"      : "pritamdeka/S-PubMedBert-MS-MARCO",
    "llm_model"            : "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "load_in_4bit"         : True,

    # ── Index paths (built by Notebook 01) ───────────────────────────
    # Adjust these paths to wherever your Kaggle dataset is mounted.
    "faiss_index_path"     : "/kaggle/input/datasets/ibtihelmhamdi/faiss-index",  
    
    # ── Retrieval ─────────────────────────────────────────────────────
    "top_k_retrieval"      : 5,
    
    # ── LLM generation ───────────────────────────────────────────────
    "max_new_tokens"       : 512,
    "temperature"          : 0.1,
    "repetition_penalty"   : 1.15,

    # ── Confidence thresholds (cosine similarity, 0–1) ───────────────
    "confidence_threshold" : 0.40,   # below → low-confidence warning
    "confidence_refuse"    : 0.20,   # below → refuse to answer

    # ── LangGraph agent ──────────────────────────────────────────────
    "max_retries"          : 2,      # rewrite-and-retry budget
}
print(" Config ready")

In [ ]:
# ── Cell 3: Embedding model + FAISS vector store ─────────────────────
#
# Loads the pre-built FAISS index from Notebook 01.


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()} for embeddings")

print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
print(f" Embeddings ready: {CONFIG["embedding_model"]}")

print(f"Loading FAISS index from: {CONFIG["faiss_index_path"]}")
vectorstore = FAISS.load_local(
    CONFIG["faiss_index_path"],
    embedding_model,
    allow_dangerous_deserialization=True,  # required for pkl loading
)
print(f" FAISS loaded — {vectorstore.index.ntotal:,} vectors")




In [ ]:
# ── Cell 4 ───────────────────
faiss_retriever = vectorstore.as_retriever(
    search_type="mmr",                         # diversity over pure similarity
    search_kwargs={
        "k"       : CONFIG["top_k_retrieval"], # final docs returned
        "fetch_k" : 20,                        # candidates before MMR reranking
        "lambda_mult": 0.7,                    # 0=max diversity, 1=max relevance
    },
)

# Sanity check
_test = faiss_retriever.get_relevant_documents("metformin blood glucose type 2 diabetes")
print(f"FAISS-MMR retriever ready — {len(_test)} docs returned for test query")
for i, d in enumerate(_test[:3]):
    print(f"   [{i+1}] {d.metadata.get('source','?')} — {d.page_content[:80].strip()}...")

In [ ]:
# ── Cell 5: Load Llama-3.1-8B-Instruct (4-bit quantized) ────────────

# NOTE: You need a Hugging Face account and must accept the Meta Llama 3.1
# license at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
# Then add your HF token to Kaggle Secrets under key "HF_TOKEN".

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    os.environ["HUGGINGFACE_TOKEN"] = hf_token
    print("✅ HF token loaded from Kaggle Secrets")
except Exception:
    hf_token = None
    print("⚠️  No HF_TOKEN found in Kaggle Secrets.")
    print("   If the model is gated, add your token at: Notebook → Add-ons → Secrets")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {CONFIG['llm_model']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["llm_model"],
    token=hf_token,
    padding_side="left",
)
# Llama 3.x uses a special end-of-turn token
tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["llm_model"],
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.eval()

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=CONFIG["max_new_tokens"],
    temperature=CONFIG["temperature"],
    repetition_penalty=CONFIG["repetition_penalty"],
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

print(f" {CONFIG['llm_model']} loaded")

In [ ]:
# ── Cell 6: Structured output helpers (Pydantic + JSON) ──────────────

class AgentDecision(BaseModel):
    """The agent's routing decision at the start of each run."""
    needs_rewrite : bool  = Field(description="True if query is vague or ambiguous")
    tool          : str   = Field(description="Tool to call: 'literature_search' or 'quick_definition'")
    reasoning     : str   = Field(description="One-sentence justification")

class RewrittenQuery(BaseModel):
    """Output of the query-rewriting node."""
    rewritten_query : str = Field(description="Specific, retrieval-optimised medical question")
    reasoning       : str = Field(description="What was vague and how it was fixed")

class GeneratedAnswer(BaseModel):
    """Output of the answer-generation node."""
    answer          : str   = Field(description="Factual answer grounded in the retrieved context")
    key_findings    : List[str] = Field(description="Up to 3 bullet-point key findings")
    has_enough_info : bool  = Field(description="False if context is insufficient to answer")

class EvaluationResult(BaseModel):
    """Output of the evaluation node."""
    confidence      : float = Field(description="Grounding score 0–1 (cosine similarity)")
    is_grounded     : bool  = Field(description="True if confidence >= threshold")
    should_retry    : bool  = Field(description="True if confidence is low and retries remain")
    verdict         : str   = Field(description="One-sentence quality verdict")


def parse_json_response(text: str, schema: type) -> dict:
    """
    Extract JSON from LLM output and validate against a Pydantic schema.
    Returns a dict on success, or a fallback dict on failure.
    """
    # Strip markdown fences if present
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    # Extract first {...} block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group())
        validated = schema(**data)
        return validated.model_dump()
    except Exception:
        return None


print(" Pydantic schemas & JSON helpers ready")

In [ ]:
# ── Cell 7: Confidence scorer & citation formatter ────────────────────
# Preserved and improved from v2.

_scorer_model = SentenceTransformer(CONFIG["embedding_model"])

MEDICAL_DISCLAIMER = (
    "\n\n---\n"
    "⚕️  This information is for research purposes only and is NOT medical advice. "
    "Always consult a qualified healthcare professional."
)

LOW_CONF_PREFIX = (
    "⚠️  Low confidence — the retrieved abstracts only partially support "
    "this answer. Treat with caution.\n\n"
)

REFUSE_MSG = (
    "I was unable to find sufficiently relevant evidence in the database "
    "to answer this question reliably. "
    "Please try rephrasing, or consult a medical professional."
)


def compute_confidence(answer: str, source_docs: list) -> float:
    """Mean cosine similarity between the answer and retrieved chunks."""
    if not answer or not source_docs:
        return 0.0
    texts       = [d.page_content for d in source_docs]
    answer_emb  = _scorer_model.encode([answer], normalize_embeddings=True)
    context_emb = _scorer_model.encode(texts,    normalize_embeddings=True)
    return float(np.mean(cosine_similarity(answer_emb, context_emb)[0]))


def format_citations(source_docs: list) -> str:
    """Structured citation block from source documents."""
    seen, lines = set(), []
    for doc in source_docs:
        key = doc.page_content[:100]
        if key in seen:
            continue
        seen.add(key)
        meta     = doc.metadata
        src      = meta.get("source",      "Unknown")
        question = meta.get("question",    "").strip()
        decision = meta.get("decision",    "").strip()
        gold     = meta.get("gold_answer", "").strip()

        line = f"  [{len(lines)+1}] {src}"
        if question:
            line += f"\n       Q: {question[:120]}"
        if decision:
            line += f" | Decision: {decision}"
        if gold:
            line += f"\n       Summary: {gold[:150]}..."
        lines.append(line)

    return "\n\nSOURCES:\n" + "\n".join(lines) if lines else ""


def apply_confidence_gate(answer: str, source_docs: list) -> tuple:
    """Apply grounding gate + disclaimer. Returns (final_answer, score)."""
    score = compute_confidence(answer, source_docs)
    if score < CONFIG["confidence_refuse"]:
        final = REFUSE_MSG
    elif score < CONFIG["confidence_threshold"]:
        final = LOW_CONF_PREFIX + answer
    else:
        final = answer
    final += MEDICAL_DISCLAIMER
    return final, score


print("✅ Confidence scorer & citation formatter ready")

In [ ]:
# ── Cell 8: LangGraph State definition ───────────────────────────────

class MedAssistState(TypedDict):
    # Input
    question        : str
    # Query rewriting
    rewritten_query : Optional[str]
    needs_rewrite   : Optional[bool]
    # Retrieval
    documents       : Optional[List[Document]]
    # Generation
    raw_answer      : Optional[str]
    key_findings    : Optional[List[str]]
    has_enough_info : Optional[bool]
    # Evaluation
    confidence      : Optional[float]
    is_grounded     : Optional[bool]
    # Control
    retry_count     : int
    tool            : Optional[str]          # 'literature_search' | 'quick_definition'
    # Output
    final_answer    : Optional[str]
    error           : Optional[str]


# Quick-reference dictionary (preserved from v2, expanded)
QUICK_DEFINITIONS = {
    "rct"                      : "Randomized Controlled Trial — participants randomly assigned to treatment or control; gold standard for establishing causality.",
    "cohort"                   : "Group of patients followed over time in an observational study; shows association, not causation.",
    "meta-analysis"            : "Statistical pooling of multiple studies; increases statistical power. Watch for heterogeneity (I²).",
    "systematic review"        : "Comprehensive structured review of all evidence on a clinical question.",
    "placebo"                  : "Inactive treatment given to control group to blind participants to treatment assignment.",
    "double blind"             : "Neither participants nor researchers know who receives treatment vs placebo.",
    "biomarker"                : "Measurable biological indicator of disease state, e.g. HbA1c for diabetes.",
    "comorbidity"              : "Presence of two or more chronic conditions in one patient simultaneously.",
    "efficacy"                 : "Ability of a treatment to produce the desired effect under controlled conditions.",
    "prevalence"               : "Total existing cases of a disease in a population at a given time.",
    "incidence"                : "Number of new cases of a disease in a population over a defined period.",
    "mortality"                : "Death rate — often expressed as deaths per 100,000 population per year.",
    "morbidity"                : "Presence or degree of illness or disease in a population.",
    "pathogenesis"             : "The biological mechanism through which a disease develops.",
    "aetiology"                : "The cause or set of causes of a disease or condition.",
    "prognosis"                : "Predicted likely outcome or course of a disease.",
    "contraindication"         : "Condition or factor that makes a treatment inadvisable.",
    "pharmacokinetics"         : "How the body absorbs, distributes, metabolises, and excretes a drug (ADME).",
    "pharmacodynamics"         : "The effects a drug has on the body and its mechanism of action.",
    "p-value"                  : "Probability of observing the result by chance if H0 is true. p<0.05 is the conventional significance threshold.",
    "confidence interval"      : "Range of values consistent with the data. A 95% CI means 95% of such intervals contain the true value.",
    "odds ratio"               : "Ratio of odds of outcome in exposed vs unexposed group. OR>1 = increased risk.",
    "hazard ratio"             : "Relative risk of an event occurring in one group vs another over time (survival analyses).",
    "number needed to treat"   : "NNT — patients that must be treated for one additional patient to benefit.",
    "sensitivity"              : "Test's ability to correctly identify true positives: TP / (TP+FN).",
    "specificity"              : "Test's ability to correctly identify true negatives: TN / (TN+FP).",
    "positive predictive value": "PPV — probability that a positive test result is a true positive.",
    "nnt"                      : "Number Needed to Treat — patients that must be treated for one additional patient to benefit.",
    "ppv"                      : "Positive Predictive Value — probability that a positive test result is a true positive.",
    "auc"                      : "Area Under the ROC Curve — overall diagnostic accuracy of a test (0.5 = chance, 1.0 = perfect).",
    "roc"                      : "Receiver Operating Characteristic — curve plotting sensitivity vs 1-specificity across all thresholds.",
}

print("✅ State definition & quick-reference dictionary ready")

In [ ]:
# ── Cell 9: Graph node implementations ───────────────────────────────

# ─────────────────────────────────────────────────────────────────────
# NODE 1: agent_decide
# Analyses the question and decides:
#   (a) whether query rewriting is needed
#   (b) which tool to call (literature_search vs quick_definition)
# ─────────────────────────────────────────────────────────────────────
def node_agent_decide(state: MedAssistState) -> dict:
    question = state["question"]
    print(f"\n[DECIDE] Question: {question[:80]}...")

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical AI routing agent. Analyse the question and return ONLY valid JSON.
No explanation, no markdown, no preamble — just the JSON object.
<|eot_id|><|start_header_id|>user<|end_header_id|>
Question: "{question}"

Return this JSON schema exactly:
{{
  "needs_rewrite": <true if question is vague, contains pronouns like 'it'/'they'/'this', or is too short to retrieve well; false otherwise>,
  "tool": <"quick_definition" if asking for a single medical/statistical term definition; "literature_search" for all other questions>,
  "reasoning": "<one sentence>"
}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    raw = llm.invoke(prompt)
    parsed = parse_json_response(raw, AgentDecision)

    if parsed is None:
        # Fallback: heuristic routing =>rule-based decision-making to decide where to send the process next.
        words = question.lower().split()
        vague_signals = {"it", "they", "this", "that", "those", "these"}
        def_signals   = {"what", "define", "definition", "mean", "meaning"}
        needs_rewrite = bool(vague_signals & set(words)) or len(words) < 5
        tool = "quick_definition" if (len(words) <= 6 and def_signals & set(words)) else "literature_search"
        print(f"  [DECIDE] JSON parse failed — using heuristic: tool={tool}, rewrite={needs_rewrite}")
        return {"needs_rewrite": needs_rewrite, "tool": tool}

    print(f"  [DECIDE] tool={parsed['tool']} | rewrite={parsed['needs_rewrite']} | {parsed['reasoning']}")
    return {"needs_rewrite": parsed["needs_rewrite"], "tool": parsed["tool"]}


# ─────────────────────────────────────────────────────────────────────
# NODE 2: rewrite_query
# Rewrites a vague question into a specific, retrieval-optimised query.
# Skipped if needs_rewrite=False.
# ─────────────────────────────────────────────────────────────────────
def node_rewrite_query(state: MedAssistState) -> dict:
    question = state["question"]
    print(f"\n[REWRITE] Original: {question}")

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a medical query optimisation expert. Rewrite vague questions into specific,
natural language medical queries suitable for semantic search.
IMPORTANT: Use plain English only. No boolean operators (AND, OR, NOT).
No brackets, no quotes, no PubMed syntax. Just a clear, specific sentence.
Return ONLY valid JSON, no other text.
<|eot_id|><|start_header_id|>user<|end_header_id|>
Vague question: "{question}"

Return this JSON exactly:
{{
  "rewritten_query": "<plain English, specific medical query — no AND/OR/brackets>",
  "reasoning": "<what was vague and how you fixed it>"
}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    raw = llm.invoke(prompt)
    parsed = parse_json_response(raw, RewrittenQuery)

    if parsed is None or not parsed.get("rewritten_query"):
        print("  [REWRITE] Failed — keeping original question")
        return {"rewritten_query": question}

    # Extra safety: strip any boolean operators that slipped through
    rewritten = parsed["rewritten_query"]
    for operator in [" AND ", " OR ", " NOT ", "[", "]"]:
        rewritten = rewritten.replace(operator, " ")
    rewritten = rewritten.strip()

    print(f"  [REWRITE] → {rewritten}")
    return {"rewritten_query": rewritten}

# ─────────────────────────────────────────────────────────────────────
# NODE 3: retrieve
# Uses rewritten_query if available, otherwise the original question.
# ─────────────────────────────────────────────────────────────────────
def node_retrieve(state: MedAssistState) -> dict:
    query = state.get("rewritten_query") or state["question"]
    print(f"\n[RETRIEVE] Query: {query[:80]}")

    try:
        docs = faiss_retriever.get_relevant_documents(query)
        print(f"  [RETRIEVE] {len(docs)} documents retrieved")
        return {"documents": docs}
    except Exception as e:
        print(f"  [RETRIEVE] Error: {e} — returning empty list")
        return {"documents": [], "error": str(e)}


# ─────────────────────────────────────────────────────────────────────
# NODE 4: quick_definition
# Handles single-term definition queries via the quick-reference dict.
# Bypasses retrieval and generation for fast, reliable answers.
# ─────────────────────────────────────────────────────────────────────
def node_quick_definition(state: MedAssistState) -> dict:
    term = state["question"].lower().strip()
    # Strip question words to find the core term
    for stopword in ["what is", "what does", "define", "definition of", "mean", "meaning of", "?"]:
        term = term.replace(stopword, "").strip()

    # Exact match
    if term in QUICK_DEFINITIONS:
        answer = f"DEFINITION — {term.upper()}:\n{QUICK_DEFINITIONS[term]}"
    else:
        # Partial match
        match = next((v for k, v in QUICK_DEFINITIONS.items() if k in term or term in k), None)
        if match:
            answer = f"DEFINITION (closest match):\n{match}"
        else:
            answer = (
                f"Term '{term}' not found in the quick-reference dictionary. "
                "Routing to medical literature search for a context-based explanation."
            )
            # Escalate to literature search
            return {"tool": "literature_search", "raw_answer": None}

    print(f"\n[QUICK DEF] {answer[:80]}...")
    return {
        "raw_answer"      : answer,
        "final_answer"    : answer + MEDICAL_DISCLAIMER,
        "confidence"      : 1.0,
        "is_grounded"     : True,
        "has_enough_info" : True,
        "key_findings"    : [answer],
    }


# ─────────────────────────────────────────────────────────────────────
# NODE 5: generate_answer
# Uses the LLM to synthesise an answer grounded in the retrieved docs.
# Returns structured JSON (GeneratedAnswer schema).
# ─────────────────────────────────────────────────────────────────────
def node_generate_answer(state: MedAssistState) -> dict:
    question = state.get("rewritten_query") or state["question"]
    docs     = state.get("documents") or []
    print(f"\n[GENERATE] Synthesising answer from {len(docs)} docs...")

    if not docs:
        return {
            "raw_answer"      : REFUSE_MSG,
            "has_enough_info" : False,
            "key_findings"    : [],
        }

    # Build context (top 3 docs to stay within context window)
    context_parts = []
    for i, doc in enumerate(docs[:3], 1):
        meta = doc.metadata
        context_parts.append(
            f"[Abstract {i}] Source: {meta.get('source','?')}\n"
            f"{doc.page_content[:600]}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are MedAssist AI, a precise medical research assistant.
Answer the question using ONLY the retrieved PubMed abstracts provided.
If the abstracts do not contain enough information, set has_enough_info to false.
Return ONLY valid JSON, no other text.
<|eot_id|><|start_header_id|>user<|end_header_id|>
RETRIEVED ABSTRACTS:
{context}

QUESTION: {question}

Return this JSON schema exactly:
{{
  "answer": "<factual answer, grounded strictly in the abstracts above>",
  "key_findings": ["<finding 1>", "<finding 2>", "<finding 3>"],
  "has_enough_info": <true if abstracts adequately address the question, false otherwise>
}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    raw = llm.invoke(prompt)
    parsed = parse_json_response(raw, GeneratedAnswer)

    if parsed is None:
        # Fallback: use raw LLM output
        print("  [GENERATE] JSON parse failed — using raw text")
        clean = re.sub(r"<\|.*?\|>", "", raw).strip()
        return {
            "raw_answer"      : clean[:1000] if clean else REFUSE_MSG,
            "has_enough_info" : bool(clean),
            "key_findings"    : [],
        }

    print(f"  [GENERATE] has_enough_info={parsed['has_enough_info']}")
    return {
        "raw_answer"      : parsed["answer"],
        "has_enough_info" : parsed["has_enough_info"],
        "key_findings"    : parsed["key_findings"],
    }


# ─────────────────────────────────────────────────────────────────────
# NODE 6: evaluate
# Computes grounding confidence and decides whether to retry.
# If confidence is low and retries remain → mark for retry.
# ─────────────────────────────────────────────────────────────────────
def node_evaluate(state: MedAssistState) -> dict:
    answer   = state.get("raw_answer") or ""
    docs     = state.get("documents") or []
    retries  = state.get("retry_count", 0)

    score = compute_confidence(answer, docs)
    is_grounded  = score >= CONFIG["confidence_threshold"]
    should_retry = (not is_grounded) and (retries < CONFIG["max_retries"])

    print(f"\n[EVALUATE] confidence={score:.3f} | grounded={is_grounded} | retry={should_retry}")

    # Build the final answer with citations and disclaimer
    gated_answer, _ = apply_confidence_gate(answer, docs)
    citations       = format_citations(docs)
    final           = gated_answer + citations

    return {
        "confidence"   : score,
        "is_grounded"  : is_grounded,
        "final_answer" : final,
        # Increment retry counter so we don't loop forever
        "retry_count"  : retries + (1 if should_retry else 0),
    }


print("✅ All graph node functions defined")

In [ ]:
# ── Cell 10: Build & compile the LangGraph ────────────────────────────
#
# GRAPH TOPOLOGY:
#
#   START
#     │
#     ▼
#   agent_decide ──► quick_definition ──► END
#     │
#     ▼  (needs_rewrite=True)
#   rewrite_query
#     │
#     ▼  (always, or skipped if rewrite=False)
#   retrieve
#     │
#     ▼
#   generate_answer
#     │
#     ▼
#   evaluate
#     │
#     ├── confidence OK  ──────────────────────────► END
#     └── confidence low + retries remain ─────────► rewrite_query
#                                                      (retry loop)

# ── Conditional edge functions ────────────────────────────────────────

def route_after_decide(state: MedAssistState) -> str:
    """After decide: go to quick_definition or to (rewrite | retrieve)."""
    if state.get("tool") == "quick_definition":
        return "quick_definition"
    if state.get("needs_rewrite"):
        return "rewrite_query"
    return "retrieve"


def route_after_quick_def(state: MedAssistState) -> str:
    """If quick_definition couldn't resolve the term, escalate."""
    if state.get("tool") == "literature_search":
        return "retrieve"   # escalated
    return END


def route_after_evaluate(state: MedAssistState) -> str:
    """After evaluate: retry with rewrite or finish."""
    score   = state.get("confidence", 1.0)
    retries = state.get("retry_count", 0)
    if score < CONFIG["confidence_threshold"] and retries < CONFIG["max_retries"]:
        print(f"  [ROUTE] Low confidence ({score:.3f}) — triggering retry #{retries}")
        return "rewrite_query"   # retry loop
    return END


# ── Build the graph ───────────────────────────────────────────────────
builder = StateGraph(MedAssistState)

# Add nodes
builder.add_node("agent_decide",    node_agent_decide) #builder.add_node("node_name", node_function)
builder.add_node("rewrite_query",   node_rewrite_query)
builder.add_node("retrieve",        node_retrieve)
builder.add_node("quick_definition",node_quick_definition)
builder.add_node("generate_answer", node_generate_answer)
builder.add_node("evaluate",        node_evaluate)

# Entry point
builder.set_entry_point("agent_decide") #always runs first

# Conditional routing after decide
builder.add_conditional_edges(
    "agent_decide", #"source_node", The node we just finished executing
    route_after_decide, #A function that looks at the state and returns a decision (a string)
    {
        "quick_definition" : "quick_definition", # "condition_1": "next_node_1",
        "rewrite_query"    : "rewrite_query",
        "retrieve"         : "retrieve",
    },
)

# quick_definition → END or escalate to retrieve
builder.add_conditional_edges(
    "quick_definition", #After node A
    route_after_quick_def, #routing_function=> A routing function looks at the current state and returns a string indicating which node to go to next.
    {
        "retrieve" : "retrieve",
        END        : END,
    },
)

# Linear edges: rewrite_query → retrieve → generate_answer → evaluate
builder.add_edge("rewrite_query",   "retrieve") #builder.add_edge("source_node", "destination_node")
builder.add_edge("retrieve",        "generate_answer")
builder.add_edge("generate_answer", "evaluate")

# Conditional routing after evaluate (retry or finish)
builder.add_conditional_edges(
    "evaluate",
    route_after_evaluate,
    {
        "rewrite_query" : "rewrite_query",
        END             : END,
    },
)

# Compile
medassist_graph = builder.compile()
print(" LangGraph compiled successfully")
print(f"   Nodes : {list(medassist_graph.get_graph().nodes.keys())}")

In [ ]:
# ── Cell 11: Main inference function ─────────────────────────────────
#
# query_medassist() is the public API.
# It wraps the LangGraph run with error handling + RAG fallback.

def direct_rag_fallback(question: str) -> str:
    """Direct RAG call used when the graph fails entirely."""
    print("  [FALLBACK] Running direct RAG...")
    try:
        docs     = faiss_retriever.get_relevant_documents(question)
        context  = "\n\n".join(d.page_content[:500] for d in docs[:3])
        prompt   = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
            f"You are MedAssist AI. Answer using ONLY the context below.\n"
            f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
            f"Context:\n{context}\n\nQuestion: {question}\n"
            f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
        )
        answer = llm.invoke(prompt).strip()
        gated, score = apply_confidence_gate(answer, docs)
        return gated + format_citations(docs) + f"\n\n[Fallback RAG | Confidence: {score:.2f}]"
    except Exception as e:
        return f"[Fallback also failed: {e}]\n" + MEDICAL_DISCLAIMER


def query_medassist(question: str, verbose: bool = True) -> dict:
    """
    Run MedAssist AI on a question.

    Parameters
    ----------
    question : str   — The medical question to answer.
    verbose  : bool  — Print node-level trace (default True).

    Returns
    -------
    dict with keys: final_answer, confidence, retry_count, tool.
    """
    sep = "=" * 65
    if verbose:
        print(f"\n{sep}")
        print(f"❓  {question}")
        print(sep)

    initial_state: MedAssistState = {
        "question"        : question,
        "rewritten_query" : None,
        "needs_rewrite"   : None,
        "documents"       : None,
        "raw_answer"      : None,
        "key_findings"    : None,
        "has_enough_info" : None,
        "confidence"      : None,
        "is_grounded"     : None,
        "retry_count"     : 0,
        "tool"            : None,
        "final_answer"    : None,
        "error"           : None,
    }

    try:
        final_state = medassist_graph.invoke(initial_state)
        answer = final_state.get("final_answer") or ""

        if not answer or len(answer.strip()) < 20:
            print("⚠️  Empty answer from graph — running fallback RAG")
            answer = direct_rag_fallback(question)
            final_state["final_answer"] = answer

    except Exception as e:
        print(f"⚠️  Graph exception: {e} — running fallback RAG")
        answer = direct_rag_fallback(question)
        final_state = {**initial_state, "final_answer": answer, "error": str(e)}

    if verbose:
        print(f"\n{'─'*65}")
        print("🤖  FINAL ANSWER:")
        print(answer)
        if final_state.get("key_findings"):
            print("\n📌  KEY FINDINGS:")
            for f in final_state["key_findings"]:
                print(f"    • {f}")
        conf = final_state.get("confidence")
        if conf is not None:
            print(f"\n📊  Confidence: {conf:.3f} | Retries: {final_state.get('retry_count', 0)} | Tool: {final_state.get('tool')}")

    return final_state


print("✅ query_medassist() ready")

In [ ]:
# ── Cell 12: Inference examples ───────────────────────────────────────
# Four representative queries — one per routing path:
#   1. Literature search (specific clinical question)
#   2. Vague query → rewrite → literature search
#   3. Single-term definition
#   4. Statistical concept definition

print("\n" + "#"*65)
print("# INFERENCE EXAMPLES")
print("#"*65)

# ── Query 1: specific clinical evidence question ──────────────────────
r1 = query_medassist(
    "What are the effects of metformin on blood glucose in type 2 diabetes?"
)

# ── Query 2: vague pronoun-heavy question ─────────────────────────────
r2 = query_medassist(
    "Does it help with sugar levels in elderly patients?"
)

# ── Query 3: single-term definition ──────────────────────────────────
r3 = query_medassist(
    "What does RCT mean?"
)

# ── Query 4: statistical concept ─────────────────────────────────────
r4 = query_medassist(
    "What does a p-value of 0.03 mean in a clinical trial?"
)

In [ ]:
# ── Cell 13: Additional inference examples ────────────────────────────

# ── Query 5: complex treatment comparison ────────────────────────────
r5 = query_medassist(
    "How does SGLT2 inhibitor therapy compare to GLP-1 receptor agonists "
    "for cardiovascular outcomes in type 2 diabetes?"
)

# ── Query 6: disease mechanism ───────────────────────────────────────
r6 = query_medassist(
    "What is the role of amyloid-beta plaques in Alzheimer's disease pathogenesis?"
)

# ── Query 7: statistical definition ─────────────────────────────────
r7 = query_medassist(
    "Define sensitivity and specificity"
)